# Linear Equation Solver (A x = b)

This notebook implements a robust solver that chooses an appropriate algorithm based on the matrix shape and rank as described below. Examples follow at the end of the notebook.


## Method selection strategy

Given A (m x n) and b (m or m x k) compute rank r = rank(A).

- If m=n=r: system is square and nonsingular → solve directly (LU or direct solver).
- If m>n=r: overdetermined with full column rank → QR-based least-squares.
- If n>m=r: underdetermined with full row rank → minimum-energy solution computed as x = A^T (A A^T)^{-1} b.
- If r < min(m, n): rank deficient → use SVD to compute a pseudoinverse and the minimal-norm least-squares solution.


In [2]:
# Implementation of the intelligent linear-system solver
import numpy as np

try:
    import scipy.linalg as sla
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def linear_solve(A, b, return_details=False, rtol=None):
    """Solve Ax = b using a method selected from matrix shape and rank.

    Parameters
    ----------
    A : array_like, shape (m, n)
        Coefficient matrix.
    b : array_like, shape (m,) or (m, k)
        Right-hand side (one or multiple columns).
    return_details : bool
        If True, return (x, details) where details is a dict with keys
        'method', 'rank', 'residual'. Otherwise return x only.
    rtol : float or None
        Optional tolerance to use for rank estimation. If None, default
        behavior of numpy.linalg.matrix_rank is used.

    Returns
    -------
    x : ndarray, shape (n,) or (n, k)
        Solution vector (or matrix for multiple RHS). Behavior depends on
        the scenario described in the notebook.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    if A.ndim != 2:
        raise ValueError("A must be 2-dimensional")

    m, n = A.shape

    # normalize b to be (m, k)
    if b.ndim == 1:
        b_was_1d = True
        if b.shape[0] != m:
            raise ValueError(f"Incompatible dimensions: A is {m}x{n}, but b has length {b.shape[0]}")
        B = b.reshape(m, 1)
    elif b.ndim == 2:
        b_was_1d = False
        if b.shape[0] != m:
            raise ValueError(f"Incompatible dimensions: A is {m}x{n}, but b has {b.shape[0]} rows")
        B = b
    else:
        raise ValueError("b must be 1D or 2D array")

    # rank
    if rtol is None:
        r = np.linalg.matrix_rank(A)
    else:
        r = np.linalg.matrix_rank(A, tol=rtol)

    eps = np.finfo(float).eps

    method = None

    # Case 1: square and full rank
    if m == n == r:
        method = "direct (LU/solve)"
        try:
            if _HAS_SCIPY:
                # use LU if available
                lu_and_piv = sla.lu_factor(A)
                X = sla.lu_solve(lu_and_piv, B)
            else:
                X = np.linalg.solve(A, B)
        except Exception:
            # fallback
            X = np.linalg.solve(A, B)

    # Case 2: overdetermined with full column rank (m > n = r)
    elif (m > n) and (r == n):
        method = "QR (least-squares)"
        Q, R = np.linalg.qr(A, mode="reduced")
        Y = Q.T @ B
        # R is square (n,n)
        X = np.linalg.solve(R, Y)

    # Case 3: underdetermined with full row rank (n > m = r)
    elif (n > m) and (r == m):
        method = "minimum-norm (Moore-Penrose formula)"
        # x = A^T (A A^T)^{-1} b
        M = A @ A.T
        # M is m x m and invertible when rank(A) == m
        Y = np.linalg.solve(M, B)
        X = A.T @ Y

    # Case 4: rank-deficient -> use SVD pseudoinverse
    else:
        method = "SVD (pseudoinverse)"
        U, s, Vt = np.linalg.svd(A, full_matrices=False)
        tol = max(A.shape) * eps * (s[0] if s.size else 0.0)
        s_inv = np.array([1.0 / si if si > tol else 0.0 for si in s])
        A_pinv = Vt.T @ np.diag(s_inv) @ U.T
        X = A_pinv @ B

    # prepare outputs
    if b_was_1d:
        x_out = X.reshape(-1)
    else:
        x_out = X

    residual = np.linalg.norm(A @ x_out - b, ord=2)
    details = {"method": method, "rank": int(r), "residual": float(residual)}

    if return_details:
        return x_out, details
    return x_out


In [3]:
# Demonstration and tests for the solver

import numpy as np

np.set_printoptions(precision=6, suppress=True)

# 1) Square & full rank (m = n = r)
rng = np.random.RandomState(1)
A = rng.randn(4,4)
b = rng.randn(4)
x, info = linear_solve(A, b, return_details=True)
print("Square full-rank -> method:", info['method'], ", residual:", info['residual'])
# compare to np.linalg.solve
x_ref = np.linalg.solve(A, b)
print("||x - x_ref|| =", np.linalg.norm(x - x_ref))

# 2) Overdetermined m>n, full column rank
A = rng.randn(6,3)
b = rng.randn(6)
x, info = linear_solve(A, b, return_details=True)
print("Overdetermined (full column rank) -> method:", info['method'], ", residual:", info['residual'])
# compare to lstsq
x_ref, *_ = np.linalg.lstsq(A, b, rcond=None)
print("||x - x_ref|| =", np.linalg.norm(x - x_ref))

# 3) Underdetermined n>m, full row rank -> minimum-norm
A = rng.randn(3,6)
b = rng.randn(3)
x, info = linear_solve(A, b, return_details=True)
print("Underdetermined (full row rank) -> method:", info['method'], ", residual:", info['residual'])
# compare to pinv
x_ref = np.linalg.pinv(A) @ b
print("||x - x_ref|| =", np.linalg.norm(x - x_ref))
print("||x|| =", np.linalg.norm(x), ", ||x_ref|| =", np.linalg.norm(x_ref))

# 4) Rank-deficient -> SVD/pinv
# make rank 2 matrix
U = rng.randn(4,2)
V = rng.randn(3,2)
A = U @ V.T  # 4x3, rank <=2
b = rng.randn(4)
x, info = linear_solve(A, b, return_details=True)
print("Rank-deficient -> method:", info['method'], ", residual:", info['residual'])
# compare to pinv
x_ref = np.linalg.pinv(A) @ b
print("||x - x_ref|| =", np.linalg.norm(x - x_ref))

Square full-rank -> method: direct (LU/solve) , residual: 5.034405433428661e-16
||x - x_ref|| = 0.0
Overdetermined (full column rank) -> method: QR (least-squares) , residual: 2.194104891311578
||x - x_ref|| = 2.633125101432526e-16
Underdetermined (full row rank) -> method: minimum-norm (Moore-Penrose formula) , residual: 1.1102230246251565e-16
||x - x_ref|| = 2.6177661246514813e-16
||x|| = 0.33722925868407133 , ||x_ref|| = 0.33722925868407155
Rank-deficient -> method: SVD (pseudoinverse) , residual: 0.5536671330343381
||x - x_ref|| = 1.1102230246251565e-16
